In [1]:
import requests
from bs4 import BeautifulSoup
import datetime
import pandas as pd
import numpy as np
import re

res = requests.get('https://registrar.ucdavis.edu/calendar/archive/academic-calendar')

# Using pd.read_html() instead

In [2]:
archivedYears = pd.read_html(
    'https://registrar.ucdavis.edu/calendar/archive/academic-calendar',
    storage_options={'User-Agent':'Mozilla/5.0'}
    )

In [3]:
currentYears = pd.read_html(
    'https://registrar.ucdavis.edu/calendar/academic-calendar',
    storage_options={'User-Agent':'Mozilla/5.0'}
    )

In [4]:
academicYears = archivedYears + currentYears

### Helper functions

In [5]:
def getInstructionDateRange(quarterDates, year):
    # Find quarter begin and end as a string
    instructionBeginStr = quarterDates[quarterDates["Desc"] == 'Instruction Begins']["Dates"].array[0]
    instructionEndsStr = quarterDates[quarterDates["Desc"] == 'Instruction Ends']["Dates"].array[0]
    
    # Add year to the string
    instructionBeginStr = instructionBeginStr + '/' + str(year)
    instructionEndsStr = instructionEndsStr + '/' + str(year)
    
    # Create datetime objects using the strings
    instructionBeginDateTime = datetime.datetime.strptime(instructionBeginStr,'%-m/%d/%Y')
    instructionEndsDateTime = datetime.datetime.strptime(instructionEndsStr,'%-m/%d/%Y')

    # Create date range using datetime objects
    instructionRange = pd.date_range(start=instructionBeginDateTime, end=instructionEndsDateTime)
    return instructionRange

Although the UC Davis calendar treats weekends not as finals, we will treat the weekends as essentially still finals because students will still be studying

In [6]:
def getFinalsDateRange(quarterDates, year):
    # Find finals begin/end as a string
    finalsStr = quarterDates[quarterDates["Desc"] == 'Finals Begin/End']["Dates"].array[0]

    # Parse month and day information based on their consistent format
    if finalsStr.find(',') != -1:
        month = int(finalsStr.split('/')[0])
        firstDay = int(finalsStr.split('/')[1].split(',')[0])
        lastDay = int(finalsStr.split('-')[1])
    else:
        month = int(finalsStr.split('/')[0])
        firstDay = int(finalsStr.split('/')[1].split('-')[0])
        lastDay = int(finalsStr.split('-')[1])

    # Create finals start and end datetime objects
    finalsStartDateTime = datetime.datetime(year=year,month=month,day=firstDay)
    finalsEndDateTime = datetime.datetime(year=year,month=month,day=lastDay)

    # Create date range using datetime objects
    finalsRange = pd.date_range(start=finalsStartDateTime, end=finalsEndDateTime)
    return finalsRange

In [7]:
def getHolidaysDateRange(quarterDates, year):
    # Initialize output
    outputDateRange = None

    # Find holidays in quarter dates
    holidays = quarterDates[quarterDates["Desc"].str.contains("Holiday")]
    for _, data in holidays.iterrows():
        dateStr = str(data["Dates"])

        # Initialize some values so that they can be accessed outside of the conditional statements
        firstDateStr = ""
        secondDateStr = ""
        firstYear = year
        secondYear = year
        
        # Find day and month of first day
        # Multiple days case
        if (dateStr.find("&") != -1) or (dateStr.find(",") != -1) or (dateStr.find("-") != -1):
            # Two dates split using '&'
            if len(dateStr.split("&")) == 2:
                firstDateStr, secondDateStr = dateStr.replace(" ","").split("&")
            # Two dates split using ','
            elif len(dateStr.split(",")) == 2:
                firstDateStr, secondDateStr = dateStr.replace(" ","").split(",")
            # Two contiguous dates denoted with '-'
            elif len(dateStr.split("-")) == 2:
                firstDateStr, secondDateStr = dateStr.replace(" ","").split("-")
            firstMonth = int(firstDateStr.split("/")[0])
            firstDay = int(firstDateStr.split("/")[1])
        # Just one day case
        else:
            firstMonth = int(dateStr.split("/")[0])
            firstDay = int(dateStr.split("/")[1])

        # Find day and month of second day
        # Second date is in a different month
        if len(secondDateStr.split("/")) == 2:
            secondMonth = int(secondDateStr.split("/")[0])
            secondDay = int(secondDateStr.split("/")[1])
            # Update secondYear if necessory
            if firstMonth > secondMonth:
                secondYear += 1
        # Second date isn't in a different month
        elif secondDateStr != "":
            secondMonth = firstMonth
            secondDay = int(secondDateStr)
        # No second date at all
        else:
            secondMonth = firstMonth
            secondDay = firstDay        

        # Convert to datetime objects
        firstDateTime = datetime.datetime(firstYear, firstMonth, firstDay)
        secondDateTime = datetime.datetime(secondYear, secondMonth, secondDay)
        
        # Create or append to output datetime range
        if outputDateRange is None:
            outputDateRange = pd.date_range(firstDateTime, secondDateTime)
        else:
            outputDateRange = outputDateRange.union(pd.date_range(firstDateTime, secondDateTime))

    return outputDateRange

### Main date-finding function for a full academic year

In [8]:
def parseAcademicYear(table):
    # Rename columns so it isn't obnoxious to look at
    table = table.rename(columns={
        table.columns[0]:'Desc',
        table.columns[2]:'Dates'
    })
    
    # Find the start and end year of each academic year (each represented by one table)
    startYear = 0
    try:
        startYear = int(re.findall(r'\d+', table.columns[1])[0])
    except:
        startYear = int(re.findall(r'\d+', table[1][0])[0])
        print("Format is too different for {startYear}, skipping this one".format(startYear=startYear))
        return None, None, None, None
    endYear = startYear + 1
    
    # Identify which quarter each row is referring to
    table["Quarter Number"] = table['Desc'].isnull().cumsum()
    
    fallDates = table[table["Quarter Number"] == 0]
    winterDates = table[table["Quarter Number"] == 1]
    springDates = table[table["Quarter Number"] == 2]
    summerDates = table[table["Quarter Number"] == 3]
    
    # Drop second column, doesn't give any really useful information
    # Exception is summer dates, where the format changed in 2023-2024
    # Instead, we drop the null column and rename second column to "Dates" again just in case
    fallDates = fallDates.drop(columns=fallDates.columns[1])
    winterDates = winterDates.drop(columns=winterDates.columns[1])
    springDates = springDates.drop(columns=springDates.columns[1])
    
    if summerDates["Dates"].isna().all():
        summerDates = summerDates.drop(columns="Dates")
        summerDates = summerDates.rename(columns=
                                     {summerDates.columns[1]:'Dates'}
                                      )
    else:
        summerDates = summerDates.drop(columns=summerDates.columns[1])

    fallInstruction = getInstructionDateRange(fallDates, startYear)
    winterInstruction = getInstructionDateRange(winterDates, endYear)
    springInstruction = getInstructionDateRange(springDates, endYear)
    instructionRange = fallInstruction.union(winterInstruction).union(springInstruction)

    fallFinals = getFinalsDateRange(fallDates, startYear)
    winterFinals = getFinalsDateRange(winterDates, endYear)
    springFinals = getFinalsDateRange(springDates, endYear)
    finalsRange = fallFinals.union(winterFinals).union(springFinals)

    fallHolidays = getHolidaysDateRange(fallDates, startYear)
    winterHolidays = getHolidaysDateRange(winterDates, endYear)
    springHolidays = getHolidaysDateRange(springDates, endYear)
    summerHolidays = getHolidaysDateRange(summerDates, endYear)
    holidaysRange = fallHolidays.union(winterHolidays).union(springHolidays).union(summerHolidays)

    return instructionRange, finalsRange, holidaysRange, startYear

In [9]:
# Initialize outputs
fullInstructionRange = None
fullFinalsRange = None
fullHolidaysRange = None

# Iterate through archived years
for table in academicYears:
    # Gather info from the academic year table
    instructionRange, finalsRange, holidaysRange, startYear = parseAcademicYear(table)

    # Skip iteration if information can't be parsed
    if instructionRange is None:
        continue

    # Construct output
    if fullInstructionRange is None:
        fullInstructionRange = instructionRange
        fullFinalsRange = finalsRange
        fullHolidaysRange = holidaysRange
    else:
        fullInstructionRange = fullInstructionRange.union(instructionRange)
        fullFinalsRange = fullFinalsRange.union(finalsRange)  
        fullHolidaysRange = fullHolidaysRange.union(holidaysRange)
    
    # print(startYear)
    # print(instructionRange)
    # print(finalsRange)
    # print(holidaysRange)

display(fullInstructionRange, fullFinalsRange, fullHolidaysRange)

Format is too different for 2017, skipping this one
Format is too different for 2016, skipping this one


DatetimeIndex(['2018-09-26', '2018-09-27', '2018-09-28', '2018-09-29',
               '2018-09-30', '2018-10-01', '2018-10-02', '2018-10-03',
               '2018-10-04', '2018-10-05',
               ...
               '2027-05-25', '2027-05-26', '2027-05-27', '2027-05-28',
               '2027-05-29', '2027-05-30', '2027-05-31', '2027-06-01',
               '2027-06-02', '2027-06-03'],
              dtype='datetime64[us]', length=1872, freq=None)

DatetimeIndex(['2018-12-10', '2018-12-11', '2018-12-12', '2018-12-13',
               '2018-12-14', '2019-03-18', '2019-03-19', '2019-03-20',
               '2019-03-21', '2019-03-22',
               ...
               '2027-03-17', '2027-03-18', '2027-03-19', '2027-06-04',
               '2027-06-05', '2027-06-06', '2027-06-07', '2027-06-08',
               '2027-06-09', '2027-06-10'],
              dtype='datetime64[us]', length=153, freq=None)

DatetimeIndex(['2018-11-12', '2018-11-22', '2018-11-23', '2018-12-24',
               '2018-12-25', '2018-12-31', '2019-01-01', '2019-01-21',
               '2019-02-18', '2019-03-29',
               ...
               '2026-12-25', '2026-12-31', '2027-01-01', '2027-01-18',
               '2027-02-15', '2027-03-26', '2027-05-31', '2027-06-18',
               '2027-07-05', '2027-09-06'],
              dtype='datetime64[us]', length=130, freq=None)

In [10]:
# Export these date ranges
pd.to_pickle(fullInstructionRange, 'instructionDates.pkl')
pd.to_pickle(fullFinalsRange, 'finalsDates.pkl')
pd.to_pickle(fullHolidaysRange, 'holidayDates.pkl')

# Old Scraper

In [11]:
soup = BeautifulSoup(res.content, 'html.parser')

# Use this to determine which HTML element is the target to scrape. In this case, everything is stored in 'tr' inside a 'div'
# print(soup.prettify())

finalDates = []
lines = []
# Find the main content container
content = soup.find('div')
if content:
    for para in content.find_all('tr'):
        lines.append(para.text.strip())
        if 'Finals Begin' in para.text:
            finalDates.append(para.text.strip())
else:
    print("No article content found.")

finalDates
cleanDates = []

# Removing the first 16 characters from each date string to get just the date information
for date in finalDates:
    cleanDates.append(date[16:])

# Initial list of all final dates WITHOUT cleaning
# cleanDates

In [12]:
import re

# This function cleans the list above and outputs a cleaned list of dates. However, we don't yet have the years, so we will need another helper function for that!
def expand_dates(date_list):
    all_days = []
    
    for entry in date_list:
        # Remove weekday text (everything before first digit)
        entry = re.sub(r'^[^0-9]+', '', entry)
        
        # Split by comma (handles special June final dates of being Friday + Monday-Thurs)
        parts = entry.split(',')
        
        month = None
        
        for part in parts:
            part = part.strip()
            
            # Match month/day-range
            match = re.match(r'(\d+)/(\d+)(?:-(\d+))?', part)
            if match:
                month = match.group(1)
                start = int(match.group(2))
                end = int(match.group(3)) if match.group(3) else start
                
                for day in range(start, end + 1):
                    all_days.append(f"{month}/{day}")
    
    return all_days

expanded = expand_dates(cleanDates)
print(expanded)

['12/9', '12/10', '12/11', '12/12', '12/13', '3/17', '3/18', '3/19', '3/20', '3/21', '6/6', '12/11', '12/12', '12/13', '12/14', '12/15', '3/18', '3/19', '3/20', '3/21', '3/22', '6/7', '12/5', '12/6', '12/7', '12/8', '12/9', '3/20', '3/21', '3/22', '3/23', '3/24', '6/9', '12/6', '12/7', '12/8', '12/9', '12/10', '3/14', '3/15', '3/16', '3/17', '3/18', '6/3', '12/14', '12/15', '12/16', '12/17', '12/18', '3/15', '3/16', '3/17', '3/18', '3/19', '6/4', '12/9', '12/10', '12/11', '12/12', '12/13', '3/16', '3/17', '3/18', '3/19', '3/20', '6/5', '12/10', '12/11', '12/12', '12/13', '12/14', '3/18', '3/19', '3/20', '3/21', '3/22', '6/7', '12/11', '12/12', '12/13', '12/14', '12/15', '3/19', '3/20', '3/21', '3/22', '3/23', '6/8', '12/5', '12/6', '12/7', '12/8', '12/9', '3/20', '3/21', '3/22', '3/23', '3/24', '6/9']


In [13]:
# sorted(set(expanded))

In [14]:
full_dates = []

current_year = None
current_quarter = None

# Unfortunately, this isn't very pretty but the only solution I found was to iterate through EVERY scraped line
for line in lines:
    
    # Detect quarter header
    header_match = re.search(r'(Fall|Winter|Spring|Summer)\s+(\d{4})', line)
    if header_match:
        current_quarter = header_match.group(1)
        current_year = int(header_match.group(2))
        continue
    
    # Find all MM/DD patterns or ranges
    matches = re.findall(r'(\d+)/(\d+)(?:-(\d+))?', line)
    
    for match in matches:
        month = int(match[0])
        start_day = int(match[1])
        end_day = int(match[2]) if match[2] else start_day
        
        for day in range(start_day, end_day + 1):
            
            year = current_year
            
            # Handle Winter year crossover
            if current_quarter == "Winter" and month == 12:
                year = current_year - 1
            
            full_dates.append(f"{month:02d}/{day:02d}/{year}")

# Remove duplicates if desired  
full_dates = sorted(set(full_dates))
# Unfortunately as a result, this list contains every date mentioned, not just finals. So, we will once again need another helper function to filter out the final dates
# At least it has the years!
# print(full_dates)

In [15]:
final_days_no_year = [
 '12/10','12/11','12/12','12/13','12/14','12/15','12/16','12/17','12/18',
 '12/5','12/6','12/7','12/8','12/9',
 '3/14','3/15','3/16','3/17','3/18','3/19','3/20','3/21','3/22','3/23','3/24',
 '6/3','6/4','6/5','6/6','6/7','6/8','6/9'
]

# normalize to zero-padded format to match full_dates
final_days_set = set(
    f"{int(m):02d}/{int(d):02d}"
    for m, d in (date.split('/') for date in final_days_no_year)
)

filtered_final_dates = [
    full_date
    for full_date in full_dates
    if full_date[:5] in final_days_set
]

filtered_final_dates = sorted(filtered_final_dates)


# Finally, after comparing final dates from expanded to the full_dates, we can extract just the final dates with years.
print(filtered_final_dates)

['03/14/2022', '03/14/2025', '03/15/2019', '03/15/2021', '03/15/2022', '03/15/2024', '03/16/2018', '03/16/2020', '03/16/2021', '03/16/2022', '03/17/2017', '03/17/2020', '03/17/2021', '03/17/2022', '03/17/2023', '03/17/2025', '03/18/2019', '03/18/2020', '03/18/2021', '03/18/2022', '03/18/2024', '03/18/2025', '03/19/2018', '03/19/2019', '03/19/2020', '03/19/2021', '03/19/2024', '03/19/2025', '03/20/2017', '03/20/2018', '03/20/2019', '03/20/2020', '03/20/2023', '03/20/2024', '03/20/2025', '03/21/2017', '03/21/2018', '03/21/2019', '03/21/2023', '03/21/2024', '03/21/2025', '03/22/2017', '03/22/2018', '03/22/2019', '03/22/2023', '03/22/2024', '03/23/2017', '03/23/2018', '03/23/2023', '03/24/2017', '03/24/2022', '03/24/2023', '06/03/2021', '06/03/2022', '06/04/2020', '06/04/2021', '06/05/2020', '06/05/2025', '06/06/2019', '06/06/2024', '06/06/2025', '06/07/2018', '06/07/2019', '06/07/2024', '06/08/2017', '06/08/2018', '06/08/2023', '06/09/2017', '06/09/2022', '06/09/2023', '12/05/2016', '12/0